In [2]:
import anndata as ad
import numpy as np
import scanpy as sc
import scanpy.external as sce
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

sc.settings.verbosity = 3

import os

In [2]:
def load_cytof_h5ad(
    file_path,
    do_arcsinh=True,
    cofactor=5,
    add_metadata=True
):
    import scanpy as sc
    import numpy as np
    import pandas as pd
    import re

    # =========================
    # 1️⃣ 读取数据
    # =========================
    adata = sc.read_h5ad(file_path)
    print(f"Loaded: {file_path}")
    print(adata)
    
    adata.obs["sample_id"] = adata.obs["sample_id"].str.replace("_raw", "", regex=False)

    # =========================
    # 2️⃣ 从 sample_id 解析 Group
    # =========================
    if add_metadata:

        if "sample_id" not in adata.obs.columns:
            raise ValueError("obs 中缺少 'sample_id' 列")

        sid = adata.obs["sample_id"].astype(str)

        # ============================================================
        # 🔴 Group 识别（关键新增）
        # ============================================================

        def infer_group(x):

            # A group
            if re.search(r"subtype_A", x):
                return "A"

            # L group（例如 V5-L116）
            if re.search(r"-L", x):
                return "L"

            # G group（如果你的命名不是 L，就改这里规则）
            if re.search(r"subtype_V[0-9]+-G", x):
                return "G"

            # fallback
            return None

        adata.obs["Group"] = sid.apply(infer_group)

        print("Group distribution:")
        print(adata.obs["Group"].value_counts())
        
        # ============================================================
        # 3️⃣ split sample_id
        # ============================================================
        split_dash = adata.obs["sample_id"].str.split("-", expand=True)

        adata.obs["part1"] = split_dash[0]
        adata.obs["part2"] = split_dash[1]
        adata.obs["part3"] = split_dash[2]

        # ============================================================
        # Group A
        # ============================================================
        mask_A = adata.obs["Group"] == "A"

        adata.obs.loc[mask_A, "Participant"] = adata.obs.loc[mask_A, "part1"]

        adata.obs.loc[mask_A, "Timepoint"] = (
            adata.obs.loc[mask_A, "part2"]
            .str.split("_")
            .str[0]
        )

        adata.obs.loc[mask_A, "Exp_id"] = (
            adata.obs.loc[mask_A, "part2"]
            .str.split("_")
            .str[1]
        )

        adata.obs.loc[mask_A, "Exp_position"] = adata.obs.loc[mask_A, "part3"]

        # ============================================================
        # Group G / L
        # ============================================================
        mask_GL = adata.obs["Group"].isin(["G", "L"])

        adata.obs.loc[mask_GL, "Timepoint"] = adata.obs.loc[mask_GL, "part1"]

        adata.obs.loc[mask_GL, "Participant"] = (
            adata.obs.loc[mask_GL, "part2"]
            .str.split("_")
            .str[0]
        )

        adata.obs.loc[mask_GL, "Exp_id"] = (
            adata.obs.loc[mask_GL, "part2"]
            .str.split("_")
            .str[1]
        )

        adata.obs.loc[mask_GL, "Exp_position"] = adata.obs.loc[mask_GL, "part3"]

        # ============================================================
        # 清理
        # ============================================================
        adata.obs.drop(
            columns=["part1", "part2", "part3"],
            inplace=True
        )
        adata.obs["Participant"] = adata.obs["Participant"].str.replace("^subtype_", "", regex=True)
        adata.obs["Timepoint"] = adata.obs["Timepoint"].str.replace("^subtype_", "", regex=True)

        # =========================
        # 4️⃣ Timepoint 映射
        # =========================
        def map_timepoint(row):

            grp = row["Group"]
            tp = row["Timepoint"]

            # ===== Group G =====
            if grp == "G" and tp == "V1":
                return "V1Day0"
            elif grp == "G" and tp == "V2":
                return "V1Day1"

            # ===== Group L =====
            elif grp == "L" and tp == "V0":
                return "V1Day0"
            elif grp == "L" and tp == "V1":
                return "V1Day1"
            elif grp == "L" and tp == "V5":
                return "V2Day0"
            elif grp == "L" and tp == "V6":
                return "V2Day1"

            # ===== Group A =====
            elif grp == "A" and tp == "V0":
                return "V1Day0"
            elif grp == "A" and tp == "V1":
                return "V1Day1"
            elif grp == "A" and tp == "V3":
                return "V2Day0"
            elif grp == "A" and tp == "V4":
                return "V2Day1"
            elif grp == "A" and tp == "V6":
                return "V1Day0"
            elif grp == "A" and tp == "V7":
                return "V1Day1"
            elif grp == "A" and tp == "V8":
                return "V2Day0"
            elif grp == "A" and tp == "V9":
                return "V2Day1"

            return None

        adata.obs["Timepoint_V"] = adata.obs.apply(map_timepoint, axis=1)
        adata.obs["GT"] = adata.obs["Group"].astype(str) + "_" + adata.obs["Timepoint_V"].astype(str)
        print("Metadata parsing finished")

    # =========================
    # 5️⃣ arcsinh
    # =========================
    if do_arcsinh:
        if "raw" not in adata.layers:
            adata.layers["raw"] = adata.X.copy()
            print("Raw layer saved")

        adata.X = np.arcsinh(adata.X / cofactor)
        print(f"arcsinh finished (cofactor={cofactor})")

    return adata

In [3]:
def plot_box(
    adata,
    cytokines,
    output_file,
    group_col="Group",
    subject_col="Participant",
    time_col="Timepoint_V",
    test_method="ttest"
):
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.stats import ttest_rel, wilcoxon
    from matplotlib.ticker import FuncFormatter
    
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    # =========================
    # 1️⃣ 提取数据
    # =========================
    df = adata.to_df()[cytokines].copy()
    
    df[subject_col] = adata.obs[subject_col].values
    df[group_col] = adata.obs[group_col].values
    df[time_col] = adata.obs[time_col].values

    # =========================
    # 2️⃣ median
    # =========================
    median_df = (
        df.groupby([subject_col, group_col, time_col], observed=False)
        .median()
        .reset_index()
    )

    long_df = median_df.melt(
        id_vars=[subject_col, group_col, time_col],
        value_vars=cytokines,
        var_name="cytokine",
        value_name="median_expr"
    )

    def get_star(p):
        if p < 1e-4:
            return "****"
        elif p < 1e-3:
            return "***"
        elif p < 1e-2:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return "ns"

    # =========================
    # 顺序
    # =========================
    group_order = ["A", "L", "G"]

    time_order = [
        "V1Day0",
        "V1Day1",
        "V2Day0",
        "V2Day1"
    ]

    # 只保留存在的
    group_order = [x for x in group_order if x in long_df[group_col].unique()]
    time_order = [x for x in time_order if x in long_df[time_col].unique()]

    # =========================
    # 3️⃣ 建图
    # =========================
    n = len(cytokines)

    ncol = 5
    nrow = 5

    fig, axes = plt.subplots(
        nrow,
        ncol,
        figsize=(ncol * 6, nrow * 4)
    )

    axes = np.array(axes).flatten()

    # =========================
    # 4️⃣ 循环画图
    # =========================
    for i, c in enumerate(cytokines):

        ax = axes[i]

        sub = long_df[
            long_df["cytokine"] == c
        ]

        # ==================================================
        # ⭐ x = Group
        # ⭐ hue = Timepoint
        # ==================================================
        sns.boxplot(
            data=sub,
            x=group_col,
            y="median_expr",
            hue=time_col,
            order=group_order,
            hue_order=time_order,
            showfliers=False,
            ax=ax,
            width=0.8,
            gap=0.1
        )

        sns.stripplot(
            data=sub,
            x=group_col,
            y="median_expr",
            hue=time_col,
            order=group_order,
            hue_order=time_order,
            dodge=True,
            palette='dark:black',
            size=2.5,
            alpha=0.5,
            ax=ax
        )

        # =========================
        # 去掉重复 legend
        # =========================
        '''
        handles, labels = ax.get_legend_handles_labels()

        n_hue = len(time_order)

        ax.legend(
            handles[:n_hue],
            labels[:n_hue],
            fontsize=7,
            frameon=False,
            title=""
        )
        '''
        ax.legend().remove()
        # =========================
        # 统计
        # =========================
        pivot = sub.pivot_table(
            index=subject_col,
            columns=[group_col, time_col],
            values="median_expr",
            observed=False
        )

        comparisons = [
            ("V1Day0", "V1Day1"),
            ("V2Day0", "V2Day1"),
        ]

        y_max = sub["median_expr"].max()
        y_min = sub["median_expr"].min()

        y_range = y_max - y_min

        if y_range == 0:
            y_range = 1e-6

        y_top = y_max + y_range * 0.25

        # ==================================================
        # 每个 group 内比较不同 timepoint
        # ==================================================
        hue_offsets = {
            "V1Day0": -0.30,
            "V1Day1": -0.10,
            "V2Day0":  0.10,
            "V2Day1":  0.30,
        }

        for g in group_order:

            for j, (t1, t2) in enumerate(comparisons):

                if ((g, t1) in pivot.columns) and ((g, t2) in pivot.columns):

                    paired = pivot[[(g, t1), (g, t2)]].dropna()

                    if len(paired) > 1:

                        try:
                            if test_method == "wilcoxon":

                                _, p = wilcoxon(
                                    paired[(g, t1)],
                                    paired[(g, t2)]
                                )

                            else:

                                _, p = ttest_rel(
                                    paired[(g, t1)],
                                    paired[(g, t2)]
                                )

                        except:
                            p = 1.0

                        star = get_star(p)

                        med1 = np.median(paired[(g, t1)])
                        med2 = np.median(paired[(g, t2)])

                        if med1 != 0:
                            fc = med2 / med1
                            label = f"{star}\n({fc:.2f}x)"
                        else:
                            label = star

                        # =========================
                        # x坐标
                        # =========================
                        x_center = group_order.index(g)

                        x1 = x_center + hue_offsets[t1]
                        x2 = x_center + hue_offsets[t2]

                        y = y_max + y_range * (0.05)

                        # =========================
                        # 画括号
                        # =========================
                        ax.plot(
                            [x1, x1, x2, x2],
                            [y, y*1.002, y*1.002, y],
                            lw=1
                        )

                        ax.text(
                            (x1 + x2) / 2,
                            y * 1.003,
                            label,
                            ha="center",
                            fontsize=7
                        )

        ax.set_title(c, fontsize=9)
        # =========================
        # ⭐ 调整 G 组在 x 轴上的视觉居中
        # =========================
        
        xticks = ax.get_xticks()
        labels = [t.get_text() for t in ax.get_xticklabels()]
        
        new_positions = xticks.copy()
        
        for idx, lab in enumerate(labels):
            if lab == "G":
                new_positions[idx] -= 0.2   # ← 向右微移（可调）
        ax.set_xticks(new_positions)
        ax.set_xticklabels(labels)
                
        ax.set_ylim(y_min, y_top)
                # =========================
        # 单独保存当前小图
        # =========================
        single_dir = os.path.join(
            os.path.dirname(output_file),
            f"{os.path.splitext(os.path.basename(output_file))[0]}_single"
        )
        os.makedirs(single_dir, exist_ok=True)

        single_file = os.path.join(
            single_dir,
            f"{c}.png"
        )

        # 当前 subplot 的边界
        extent = ax.get_tightbbox(
            fig.canvas.get_renderer()
        ).transformed(
            fig.dpi_scale_trans.inverted()
        )

        fig.savefig(
            single_file,
            bbox_inches=extent.expanded(1.05, 1.15),
            dpi=300
        )
        
    # =========================
    # 删除空白图
    # =========================
    for k in range(i + 1, len(axes)):
        fig.delaxes(axes[k])

    plt.tight_layout()

    # =========================
    # 保存
    # =========================
    plt.savefig(output_file, dpi=300)

    plt.close()

    print("Saved:", output_file)

In [3]:
def plot_box(
    adata,
    cytokines,
    output_file,
    group_col="Group",
    subject_col="Participant",
    time_col="Timepoint_V",
    test_method="ttest"
):
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.stats import ttest_rel, wilcoxon

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    # =========================
    # 1️⃣ 提取数据
    # =========================
    df = adata.to_df()[cytokines].copy()
    df[subject_col] = adata.obs[subject_col].values
    df[group_col] = adata.obs[group_col].values
    df[time_col] = adata.obs[time_col].values

    # =========================
    # 2️⃣ median
    # =========================
    median_df = (
        df.groupby([subject_col, group_col, time_col], observed=False)
        .median()
        .reset_index()
    )

    long_df = median_df.melt(
        id_vars=[subject_col, group_col, time_col],
        value_vars=cytokines,
        var_name="cytokine",
        value_name="median_expr"
    )

    def get_star(p):
        if p < 1e-4:
            return "****"
        elif p < 1e-3:
            return "***"
        elif p < 1e-2:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return "ns"

    # =========================
    # 顺序
    # =========================
    group_order = ["A", "L", "G"]
    time_order = ["V1Day0", "V1Day1", "V2Day0", "V2Day1"]

    group_order = [x for x in group_order if x in long_df[group_col].unique()]
    time_order = [x for x in time_order if x in long_df[time_col].unique()]

    # =========================
    # 共享绘图函数（核心）
    # =========================
    def draw(ax, sub, c):

        sns.boxplot(
            data=sub,
            x=group_col,
            y="median_expr",
            hue=time_col,
            order=group_order,
            hue_order=time_order,
            showfliers=False,
            ax=ax,
            width=0.8,
            gap=0.1
        )

        sns.stripplot(
            data=sub,
            x=group_col,
            y="median_expr",
            hue=time_col,
            order=group_order,
            hue_order=time_order,
            dodge=True,
            palette="dark:black",
            size=2.5,
            alpha=0.5,
            ax=ax
        )

        ax.legend().remove()
        ax.set_title(c, fontsize=9)

        # =========================
        # stats
        # =========================
        pivot = sub.pivot_table(
            index=subject_col,
            columns=[group_col, time_col],
            values="median_expr",
            observed=False
        )

        comparisons = [("V1Day0", "V1Day1"), ("V2Day0", "V2Day1")]

        y_max = sub["median_expr"].max()
        y_min = sub["median_expr"].min()
        y_range = (y_max - y_min) if (y_max != y_min) else 1e-6
        y_top = y_max + y_range * 0.25

        hue_offsets = {
            "V1Day0": -0.30,
            "V1Day1": -0.10,
            "V2Day0":  0.10,
            "V2Day1":  0.30,
        }

        for g in group_order:
            for t1, t2 in comparisons:

                if ((g, t1) in pivot.columns) and ((g, t2) in pivot.columns):

                    paired = pivot[[(g, t1), (g, t2)]].dropna()
                    if len(paired) > 1:

                        try:
                            if test_method == "wilcoxon":
                                _, p = wilcoxon(paired[(g, t1)], paired[(g, t2)])
                            else:
                                _, p = ttest_rel(paired[(g, t1)], paired[(g, t2)])
                        except:
                            p = 1.0

                        star = get_star(p)

                        med1 = np.median(paired[(g, t1)])
                        med2 = np.median(paired[(g, t2)])
                        fc = (med2 / med1) if med1 != 0 else 0

                        label = f"{star}\n({fc:.2f}x)" if med1 != 0 else star

                        x_center = group_order.index(g)
                        x1 = x_center + hue_offsets[t1]
                        x2 = x_center + hue_offsets[t2]

                        y = y_max + y_range * 0.05

                        ax.plot([x1, x1, x2, x2], [y, y*1.002, y*1.002, y], lw=1)
                        ax.text((x1 + x2)/2, y*1.003, label, ha="center", fontsize=7)

        ax.set_ylim(y_min, y_top)

        # G shift
        xticks = ax.get_xticks()
        labels = [t.get_text() for t in ax.get_xticklabels()]

        new_positions = xticks.copy()
        for idx, lab in enumerate(labels):
            if lab == "G":
                new_positions[idx] -= 0.2

        ax.set_xticks(new_positions)
        ax.set_xticklabels(labels)

    # =========================
    # 3️⃣ 大图
    # =========================
    ncol, nrow = 5, 5
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 6, nrow * 4))
    axes = np.array(axes).flatten()

    for i, c in enumerate(cytokines):

        ax = axes[i]
        sub = long_df[long_df["cytokine"] == c]

        draw(ax, sub, c)

    for k in range(i + 1, len(axes)):
        fig.delaxes(axes[k])

    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    plt.close(fig)

    # =========================
    # 4️⃣ 小图（重新生成 fig，但共享 draw函数）
    # =========================
    single_dir = os.path.join(
        os.path.dirname(output_file),
        f"{os.path.splitext(os.path.basename(output_file))[0]}_single"
    )
    os.makedirs(single_dir, exist_ok=True)

    for c in cytokines:

        sub = long_df[long_df["cytokine"] == c]

        fig_s, ax_s = plt.subplots(figsize=(6, 4))
        draw(ax_s, sub, c)

        fig_s.tight_layout()
        fig_s.savefig(os.path.join(single_dir, f"{c}.png"), dpi=300)
        plt.close(fig_s)

    print("Saved:", output_file)

In [4]:
def plot_hist(
    adata,
    cytokines,
    output_file,
    group_col="Group",
    time_col="Timepoint_V",
    group_order=None,
    time_order=None,
    colors=None,
    bins=600
):
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import os

    # =========================
    # 参数
    # =========================
    if group_order is None:
        group_order = ["A", "L", "G"]

    if time_order is None:
        time_order = [
            "V1Day0",
            "V1Day1",
            "V2Day0",
            "V2Day1"
        ]

    if colors is None:
        colors = {
            "V1Day0": "red",
            "V1Day1": "blue",
            "V2Day0": "green",
            "V2Day1": "orange"
        }

    # =========================
    # 数据
    # =========================
    adata.X = np.arcsinh(adata.X / 5)


    df = adata.to_df()[cytokines].copy()

    df[group_col] = adata.obs[group_col].values
    df[time_col] = adata.obs[time_col].values

    # =========================
    # 建图
    # 每个 cytokine 占 3 行
    # =========================
    n = len(cytokines)

    fig, axes = plt.subplots(
        nrows=n * 3,
        ncols=1,
        figsize=(8, n * 6),
        sharex=False
    )

    # 单 cytokine 特殊处理
    if n == 1:
        axes = np.array(axes).reshape(3)

    # =========================
    # 循环 cytokine
    # =========================
    for i, cytokine in enumerate(cytokines):

        # =========================
        # 全局范围
        # xmin = 0.01 quantile
        # xmax = max
        # =========================
        all_data = df[cytokine].dropna()

        xmin = np.quantile(all_data, 0.001)
        xmax = all_data.max()

        # =========================
        # 每个 group 一行
        # =========================
        for j, g in enumerate(group_order):

            ax = axes[i * 3 + j]

            sub_group = df[
                df[group_col] == g
            ]

            # =========================
            # 每个 timepoint
            # =========================
            for tp in time_order:

                sub = sub_group[
                    sub_group[time_col] == tp
                ]

                data = sub[cytokine].dropna()

                if len(data) == 0:
                    continue

                # =========================
                # 填充
                # =========================
                ax.hist(
                    data,
                    bins=bins,
                    alpha=0.25,
                    color=colors[tp],
                    density=True
                )

                # =========================
                # 轮廓
                # =========================
                ax.hist(
                    data,
                    bins=bins,
                    histtype="step",
                    linewidth=1.5,
                    color=colors[tp],
                    density=True
                )

            # =========================
            # 标题
            # =========================
            ax.set_title(
                f"{cytokine} | Group {g}",
                fontsize=10
            )

            # =========================
            # x轴范围
            # =========================
            ax.set_xlim(xmin, xmax)

            # =========================
            # legend
            # =========================
            if i == 0 and j == 0:

                handles = [
                    plt.Line2D(
                        [0],
                        [0],
                        color=colors[tp],
                        lw=2
                    )
                    for tp in time_order
                ]

                ax.legend(
                    handles,
                    time_order,
                    frameon=False,
                    fontsize=8,
                    loc="upper right"
                )

    plt.tight_layout()

    # =========================
    # 保存
    # =========================
    os.makedirs(
        os.path.dirname(output_file),
        exist_ok=True
    )

    plt.savefig(output_file, dpi=300)

    plt.close()

    print("Saved figure to:", output_file)

In [ ]:
import os

input_dir = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined_metedata/"
output_dir = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot"

os.makedirs(output_dir, exist_ok=True)

cytokine = [
    'IFN-a_144Nd','CXCL9_146Nd','CD86_147Sm','H3K27ac_148Nd',
    'IL-1b_149Sm','4E-BP1_150Nd','pS6_151Eu','CCR2_152Sm',
    'TNF-a_153Eu','IL-4_154Sm','CASP-8_156Gd','CD38_159Tb',
    'CD123_161Dy','Ki-67_162Dy','CXCL10_163Dy','Arginase-1_164Dy',
    'IL-6_165Ho','MCP-1_166Er','CD69_167Er','STC1_168Er',
    'GZMB_169Tm','HLA-DR_171Yb','IFN-g_172Yb','pSTAT1_175Lu','pSTAT3_176Yb'
]

for fname in os.listdir(input_dir):
    if not fname.endswith(".h5ad"):
        continue

    file_path = os.path.join(input_dir, fname)
    name = os.path.splitext(fname)[0]

    print("Processing:", name)

    try:
        #adata = load_cytof_h5ad(file_path,do_arcsinh= False )
        adata = sc.read_h5ad(file_path)
        # =========================
        # ⭐ boxplot（大图 PNG）
        # =========================
        box_file = os.path.join(output_dir, f"{name}_box.png")

        plot_box(   # ← 你已经是大图版本
            adata,
            cytokine,
            box_file
        )
        
        '''
        # =========================
        # ⭐ histogram（PNG）
        # =========================
        hist_file = os.path.join(output_dir, f"{name}_hist.png")

        plot_hist(
            adata,
            cytokine,
            hist_file
        )
        '''
    except Exception as e:
        print("❌ Failed:", name, "|", e)
print("✅ All done!")

Processing: CD8TCM
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/CD8TCM_box.png
Processing: NKdim
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/NKdim_box.png
Processing: CD4TEMRA
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/CD4TEMRA_box.png
Processing: CD8TEMRA


In [6]:
import os

input_dir = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined_metedata_arcsinh/"
output_dir = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined_arcsinh_plot"

os.makedirs(output_dir, exist_ok=True)

cytokine = [
    'IFN-a_144Nd','CXCL9_146Nd','CD86_147Sm','H3K27ac_148Nd',
    'IL-1b_149Sm','4E-BP1_150Nd','pS6_151Eu','CCR2_152Sm',
    'TNF-a_153Eu','IL-4_154Sm','CASP-8_156Gd','CD38_159Tb',
    'CD123_161Dy','Ki-67_162Dy','CXCL10_163Dy','Arginase-1_164Dy',
    'IL-6_165Ho','MCP-1_166Er','CD69_167Er','STC1_168Er',
    'GZMB_169Tm','HLA-DR_171Yb','IFN-g_172Yb','pSTAT1_175Lu','pSTAT3_176Yb'
]

for fname in os.listdir(input_dir):
    if not fname.endswith(".h5ad"):
        continue

    file_path = os.path.join(input_dir, fname)
    name = os.path.splitext(fname)[0]

    print("Processing:", name)

    try:
        #adata = load_cytof_h5ad(file_path,do_arcsinh= False )
        adata = sc.read_h5ad(file_path)
        # =========================
        # ⭐ boxplot（大图 PNG）
        # =========================
        box_file = os.path.join(output_dir, f"{name}_box.png")

        plot_box(   # ← 你已经是大图版本
            adata,
            cytokine,
            box_file
        )
        
        '''
        # =========================
        # ⭐ histogram（PNG）
        # =========================
        hist_file = os.path.join(output_dir, f"{name}_hist.png")

        plot_hist(
            adata,
            cytokine,
            hist_file
        )
        '''
    except Exception as e:
        print("❌ Failed:", name, "|", e)
print("✅ All done!")

Processing: CD8TCM
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/CD8TCM_box.png
Processing: NKdim
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/NKdim_box.png
Processing: CD4TEMRA
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/CD4TEMRA_box.png
Processing: CD8TEMRA
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/CD8TEMRA_box.png
Processing: T
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/T_box.png
Processing: CD4T
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/CD4T_box.png
Processing: Toltal_T
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/Toltal_T_box.png
Processing: Non_classical_Mono
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/Non_classical_Mono_box.png
Processing: gdT
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/gdT_box.png
Processing: CD4TCM
Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combi

# test

In [22]:
file_path = '/public/users/xueyupeng/VZV/cytof/subtype_combined/CD8TCM.h5ad'
adata = load_cytof_h5ad(file_path,do_arcsinh= 0 )
adata.obs

Loaded: /public/users/xueyupeng/VZV/cytof/subtype_combined/CD8TCM.h5ad
AnnData object with n_obs × n_vars = 6926190 × 52
    obs: 'sample_id'
Group distribution:
Group
A    3209966
L    2583068
G    1133156
Name: count, dtype: int64
Metadata parsing finished


,sample_id,Group,Participant,Timepoint,Exp_id,Exp_position,Timepoint_V,GT
cell_0,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
cell_1,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
cell_2,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
cell_3,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
cell_4,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
...,...,...,...,...,...,...,...,...
cell_6926185,subtype_A005-V3_20260129-1_7-3,A,A005,V3,20260129,1_7,V2Day0,A_V2Day0
cell_6926186,subtype_A005-V3_20260129-1_7-3,A,A005,V3,20260129,1_7,V2Day0,A_V2Day0
cell_6926187,subtype_A005-V3_20260129-1_7-3,A,A005,V3,20260129,1_7,V2Day0,A_V2Day0
cell_6926188,subtype_A005-V3_20260129-1_7-3,A,A005,V3,20260129,1_7,V2Day0,A_V2Day0


In [27]:
file_path = '/public/users/xueyupeng/VZV/cytof/subtype_raw_combined/CD8TCM.h5ad'
adata = load_cytof_h5ad(file_path,do_arcsinh= 0 )
adata.obs

Loaded: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined/CD8TCM.h5ad
AnnData object with n_obs × n_vars = 6926190 × 52
    obs: 'sample_id'
Group distribution:
Group
A    3209966
L    2583068
G    1133156
Name: count, dtype: int64
Metadata parsing finished


,sample_id,Group,Participant,Timepoint,Exp_id,Exp_position,Timepoint_V,GT
cell_0,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
cell_1,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
cell_2,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
cell_3,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
cell_4,subtype_V5-L116_20251127-4_7-3,L,L116,V5,20251127,4_7,V2Day0,L_V2Day0
...,...,...,...,...,...,...,...,...
cell_6926185,subtype_A005-V3_20260129-1_7-3,A,A005,V3,20260129,1_7,V2Day0,A_V2Day0
cell_6926186,subtype_A005-V3_20260129-1_7-3,A,A005,V3,20260129,1_7,V2Day0,A_V2Day0
cell_6926187,subtype_A005-V3_20260129-1_7-3,A,A005,V3,20260129,1_7,V2Day0,A_V2Day0
cell_6926188,subtype_A005-V3_20260129-1_7-3,A,A005,V3,20260129,1_7,V2Day0,A_V2Day0


In [29]:
adata.obs.Timepoint.unique()

array(['V5', 'V6', 'V0', 'V2', 'V1', 'V4', 'V9', 'V8', 'V3', 'V7'],
      dtype=object)

In [35]:
input_dir = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined"
output_dir = "/public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot"

os.makedirs(output_dir, exist_ok=True)

cytokine = [
    'IFN-a_144Nd','CXCL9_146Nd','CD86_147Sm','H3K27ac_148Nd',
    'IL-1b_149Sm','4E-BP1_150Nd','pS6_151Eu','CCR2_152Sm',
    'TNF-a_153Eu','IL-4_154Sm','CASP-8_156Gd','CD38_159Tb',
    'CD123_161Dy','Ki-67_162Dy','CXCL10_163Dy','Arginase-1_164Dy',
    'IL-6_165Ho','MCP-1_166Er','CD69_167Er','STC1_168Er',
    'GZMB_169Tm','HLA-DR_171Yb','IFN-g_172Yb','pSTAT1_175Lu','pSTAT3_176Yb'
]
name = 'CD8TCM'
box_file = os.path.join(output_dir, f"{name}_box.png")

 
plot_box(   # ← 你已经是大图版本
    adata,
    cytokine,
    box_file)

Saved: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/CD8TCM_box.png


In [42]:
hist_file = os.path.join(output_dir, f"{name}_hist.png")

plot_hist(
    adata,
    cytokine,
    hist_file
)

Saved figure to: /public/users/xueyupeng/VZV/cytof/subtype_raw_combined_plot/CD8TCM_hist.png


In [15]:
adata.var

""
CD45-1_106Cd
CD45-2_110Cd
CD45-3_111Cd
CD8_112Cd
CD3_113In
CD45-4_114Cd
CD4_115In
CD45-5_116Cd
CD57_139La
CD56_141Pr


## 每个样本随机取100个细胞做测试

In [ ]:
import numpy as np
import scanpy as sc

# =========================
# 每个 sample 随机抽取 100 个细胞
# =========================
def downsample_per_sample(
    adata,
    sample_col="sample_id",
    n_cells=100,
    random_state=0
):

    np.random.seed(random_state)

    keep_idx = []

    # 遍历每个 sample
    for sample in adata.obs[sample_col].unique():

        idx = np.where(
            adata.obs[sample_col] == sample
        )[0]

        # 如果细胞数 <= n_cells，全部保留
        if len(idx) <= n_cells:

            keep_idx.extend(idx)

        else:

            chosen = np.random.choice(
                idx,
                size=n_cells,
                replace=False
            )

            keep_idx.extend(chosen)

    # 排序（可选）
    keep_idx = np.sort(keep_idx)

    # subset
    adata_sub = adata[keep_idx].copy()

    print("Original cells:", adata.n_obs)
    print("Downsampled cells:", adata_sub.n_obs)

    return adata_sub

## 显示数据分布

In [28]:
import pandas as pd

df = adata.to_df()

summary = pd.DataFrame({
    "min": df.min(),
    "max": df.max(),
    "mean": df.mean(),
    "median": df.median(),
    "std": df.std()
})

print(summary)

                           min           max         mean       median  \
CD45-1_106Cd      2.008721e-05  2.737285e+04  1569.551270   203.290558   
CD45-2_110Cd      2.010236e-05  2.665865e+04   910.149109   103.048111   
CD45-3_111Cd      5.369982e-05  3.155974e+04  1877.722656   376.348877   
CD8_112Cd         4.201950e+02  1.010338e+05  4264.102051  3985.195312   
CD3_113In         3.032930e+02  1.490258e+04  2388.100830  2174.068115   
CD45-4_114Cd      1.152330e-05  2.191548e+04   973.315674   316.459259   
CD4_115In         1.787519e-04  6.088909e+02   109.783699    94.754372   
CD45-5_116Cd      3.577398e-05  3.669699e+04  1583.392334    81.572533   
CD57_139La        1.070030e-05  9.886834e+04   303.038757    33.717731   
CD56_141Pr        1.056258e-05  1.580557e+04    32.997356    21.104307   
IgD_142Nd         3.014132e-06  1.609956e+04    19.059570    15.206238   
IFN-a_144Nd       1.708169e-05  7.503505e+04    23.867998    19.541027   
CXCL9_146Nd       4.043466e-06  9.8113

In [2]:
import numpy as np
import pandas as pd
from fcsparser import parse

# =====================================================
# FCS 文件
# =====================================================

fcs_file = "/public/users/xueyupeng/VZV/cytof/data/20260105/rawdata/20260105-1.fcs"

# =====================================================
# 读取（不做任何转换）
# =====================================================

meta, data = parse(
    fcs_file,
    reformat_meta=True
)

print(data.shape)

# =====================================================
# 查看每个通道范围
# =====================================================

summary = pd.DataFrame({
    "Channel": data.columns,
    "Min": data.min().values,
    "Max": data.max().values,
    "Mean": data.mean().values,
    "Median": data.median().values,
    "SD": data.std().values
})

# 排序（可选）
summary = summary.sort_values("Max", ascending=False)

# 显示
print(summary)

# =====================================================
# 保存
# =====================================================

save_path = "/public/users/xueyupeng/VZV/cytof/data/20260105/channel_raw_range.csv"

summary.to_csv(save_path, index=False)

print(f"\n✓ Saved: {save_path}")

(3075821, 53)
             Channel           Min           Max          Mean       Median  \
50            Offset -1.968637e+01  3.079274e+08  69721.523438   657.705078   
49            Center -2.606599e+05  2.011824e+05   3079.083496  3002.780762   
48             Width -1.142094e+05  1.422781e+05    245.403000   183.329620   
51          Residual -1.697455e+01  1.343036e+05    363.799469   299.657013   
45        DNA2_193Ir  2.092749e-03  7.139392e+04   1907.877319  1693.410156   
39       IFN-g_172Yb  4.427831e-05  6.420990e+04     76.305496    51.584202   
47        CD16_209Bi  5.210394e-05  6.398039e+04    109.069565    30.722464   
21       TNF-a_153Eu  4.522366e-07  6.232970e+04     83.355194    39.995926   
26        CD38_159Tb  1.228022e-04  6.112889e+04    587.583679   173.316483   
38      HLA-DR_171Yb  4.688812e-05  6.054657e+04    982.480042   137.337036   
19         pS6_151Eu  5.900310e-04  5.983229e+04   4537.737793  3565.554688   
36        GZMB_169Tm  2.346503e-02  5.

In [3]:
file_path = '/public/users/xueyupeng/VZV/cytof/subtype_raw_combined_metedata_arcsinh/Plasmablast.h5ad'
adata = sc.read_h5ad(file_path)
import pandas as pd

df = adata.to_df()

summary = pd.DataFrame({
    "min": df.min(),
    "max": df.max(),
    "mean": df.mean(),
    "median": df.median(),
    "std": df.std()
})

print(summary)

                       min        max      mean    median       std
CD45-1_106Cd      0.000572   9.036747  5.043709  5.142333  2.123682
CD45-2_110Cd      0.000169   9.045382  3.936396  3.718178  2.093065
CD45-3_111Cd      0.000108   9.096860  5.166042  5.078260  2.156783
CD8_112Cd         0.000355   8.740220  3.475378  3.547508  1.343418
CD3_113In         0.000191   5.252076  2.833371  2.875752  0.853189
CD45-4_114Cd      0.000144   8.751531  4.729077  4.846866  1.879977
CD4_115In         0.000379   7.934101  2.127472  2.181086  0.803438
CD45-5_116Cd      0.000189   9.214659  4.367850  3.518501  2.447230
CD57_139La        0.000276   9.626270  2.709808  2.648980  1.105839
CD56_141Pr        0.000056   9.741835  2.448113  2.355169  1.141634
IgD_142Nd         0.000024   9.995345  2.235848  2.011462  1.504882
IFN-a_144Nd       0.000030   9.402925  2.452966  2.561172  0.935276
CXCL9_146Nd       0.001943   9.885996  3.183199  3.262914  0.906775
CD86_147Sm        0.000158   9.266119  2.294277 

In [4]:
file_path = '/public/users/xueyupeng/VZV/cytof/subtype_raw_combined_metedata/Plasmablast.h5ad'
adata = sc.read_h5ad(file_path)
import pandas as pd

df = adata.to_df()

summary = pd.DataFrame({
    "min": df.min(),
    "max": df.max(),
    "mean": df.mean(),
    "median": df.median(),
    "std": df.std()
})

print(summary)

                       min        max      mean    median       std
CD45-1_106Cd      0.000572   9.036747  5.043709  5.142333  2.123682
CD45-2_110Cd      0.000169   9.045382  3.936396  3.718178  2.093065
CD45-3_111Cd      0.000108   9.096860  5.166042  5.078260  2.156783
CD8_112Cd         0.000355   8.740220  3.475378  3.547508  1.343418
CD3_113In         0.000191   5.252076  2.833371  2.875752  0.853189
CD45-4_114Cd      0.000144   8.751531  4.729077  4.846866  1.879977
CD4_115In         0.000379   7.934101  2.127472  2.181086  0.803438
CD45-5_116Cd      0.000189   9.214659  4.367850  3.518501  2.447230
CD57_139La        0.000276   9.626270  2.709808  2.648980  1.105839
CD56_141Pr        0.000056   9.741835  2.448113  2.355169  1.141634
IgD_142Nd         0.000024   9.995345  2.235848  2.011462  1.504882
IFN-a_144Nd       0.000030   9.402925  2.452966  2.561172  0.935276
CXCL9_146Nd       0.001943   9.885996  3.183199  3.262914  0.906775
CD86_147Sm        0.000158   9.266119  2.294277 